---
jupytext:
  text_representation:
    format_name: myst
kernelspec:
  display_name: Python 3
  language: python
  name: python3
---
# Lesson 1 Ã¢â‚¬â€ Google Colab and Your First Hydrograph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mr-pablinho/drops-course/blob/main/book/01-python-for-hydrology/01-colab-intro.ipynb)

**Learning objectives**  
By the end of this lesson you will be able to:
- [ ] Open a notebook in Google Colab, run cells, and understand cell execution order
- [ ] Fetch daily streamflow records from the USGS National Water Information System (NWIS) using the `dataretrieval` library
- [ ] Plot a labelled hydrograph and identify flood peaks, seasonal patterns, and low-flow periods


In [ ]:
# Install course dependencies when running on Google Colab
import sys
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', 'https://raw.githubusercontent.com/mr-pablinho/drops-course/main/requirements-colab.txt'
    ], check=True)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [ ]:
CI = os.environ.get('HYDRO_ML_CI', '0') == '1'

if CI:
    # Pre-fetched sample Ã¢â‚¬â€ used by CI to avoid live API calls
    df_raw = pd.read_parquet('../../data/samples/usgs_06803495_daily_2000_2015.parquet')
else:
    import dataretrieval.nwis as nwis
    raw, _ = nwis.get_dv(
        sites='06803495',
        parameterCd='00060',
        start='2000-01-01',
        end='2015-12-31',
    )
    df_raw = raw.reset_index()

# Normalize to a consistent schema regardless of source
df = (
    df_raw
    .rename(columns={'00060_Mean': 'discharge_cfs'})
    .assign(datetime=lambda d: pd.to_datetime(d['datetime']))
    .set_index('datetime')
    .sort_index()
    [['discharge_cfs']]
    .pipe(lambda d: d[d['discharge_cfs'] > 0])
    .dropna()
)

print(f'CI mode: {CI}')
print(f'Loaded {len(df):,} daily records  |  {df.index.min().date()} -> {df.index.max().date()}')
df.head()

## Introduction

On 14 June 2011, the Missouri River at Omaha, Nebraska recorded a peak discharge of
approximately **970,000 cubic feet per second** Ã¢â‚¬â€ the highest in the gauge's recorded history.
The 2011 Missouri River Flood lasted nearly six months, forced evacuations along 500 miles of
riverbank, and put two nuclear power plants into emergency flood protocols.

That event is captured in the very dataset you are about to load.
By the end of this lesson you will have produced a hydrograph that shows exactly where that
catastrophic peak sits in a 15-year streamflow record Ã¢â‚¬â€ and you will have built the Python
skills to do the same for any gauged river in the United States.


## 1. Getting Around Google Colab

Google Colab is a free, cloud-hosted Jupyter environment Ã¢â‚¬â€ no local installation required.
Each notebook runs in a virtual machine with Python and the major data-science libraries
pre-installed. You write and run code in **cells**; results appear directly below each cell.

**Essential keyboard shortcuts**

| Action | Shortcut |
|---|---|
| Run current cell and move to next | **Shift + Enter** |
| Run current cell, stay on it | **Ctrl + Enter** |
| Insert cell below | **Esc Ã¢â€ â€™ B** |
| Switch cell to Markdown | **Esc Ã¢â€ â€™ M** |
| Open command palette | **Ctrl + Shift + P** |

**Cell execution order matters.**  
Variables persist in memory across cells in the order they are run, not the order they
appear on screen. If you restart the runtime, all variables are erased and you need to
re-run from the top. A number in brackets like `[3]` shows when a cell was last run;
an asterisk `[*]` means it is still running.

> **Try it now:** press **Shift + Enter** on the code cell above to load the data,
> then continue reading here.


## 2. What Is a Hydrograph?

A **hydrograph** is a time-series plot of streamflow (discharge) at a single gauging station.
Discharge Ã¢â‚¬â€ measured in cubic feet per second (cfs) or cubic metres per second (mÃ‚Â³/s) Ã¢â‚¬â€
records how much water passes a fixed cross-section of a river per unit time.

A typical **storm hydrograph** has three characteristic phases:

1. **Rising limb** Ã¢â‚¬â€ rapid increase as surface runoff and lateral inflow reach the channel
2. **Peak discharge** Ã¢â‚¬â€ maximum flow, usually hours to days after the precipitation event
3. **Recession limb** Ã¢â‚¬â€ slower decline as groundwater drainage sustains baseflow

The cell below draws a schematic to make these features concrete before we look at real data.


In [ ]:
t = np.linspace(0, 14, 500)
q_base = 7 + 2.5 * np.exp(-0.22 * t)
q_storm = 46 * np.exp(-0.5 * ((t - 4.2) / 1.3) ** 2)
q = q_base + q_storm

peak_i = np.argmax(q)
pt, pq = t[peak_i], q[peak_i]

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(t, q, alpha=0.15, color='steelblue')
ax.plot(t, q, color='steelblue', lw=2.5, label='Streamflow Q(t)')
ax.plot(t, q_base, color='gray', lw=1.3, ls='--', label='Baseflow')

kw = dict(arrowstyle='->', lw=1.2)
ax.annotate('Rising limb', xy=(2.8, 30), xytext=(0.4, 45),
            arrowprops={**kw, 'color': 'steelblue'}, fontsize=9)
ax.annotate('Peak discharge', xy=(pt, pq + 0.4), xytext=(5.8, 57),
            arrowprops={**kw, 'color': 'navy'}, fontsize=9, color='navy')
ax.annotate('Recession limb\n(baseflow-dominated)', xy=(8.5, 19), xytext=(9.3, 35),
            arrowprops={**kw, 'color': 'steelblue'}, fontsize=9)

ax.set_xlabel('Time after rainfall event', fontsize=11)
ax.set_ylabel('Discharge Q', fontsize=11)
ax.set_title('Anatomy of a Storm Hydrograph', fontsize=13)
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlim(0, 14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 3. Loading Real Streamflow Data from USGS

The **United States Geological Survey (USGS)** operates over 8,000 active stream gauges and
publishes data through the **National Water Information System (NWIS)**.
We access NWIS programmatically using the
[`dataretrieval`](https://doi.org/10.5066/P9X4L3GE) library Ã¢â‚¬â€ developed by the USGS itself.

**Our gauge: 06803495 Ã¢â‚¬â€ Missouri River at Omaha, NE**  
The Missouri is the longest river in North America (3,767 km) and one of the most
intensively monitored. USGS parameter code **00060** gives daily mean discharge in cfs.

The data was already loaded in the cell at the top of this notebook.
Let us now examine its structure.


In [ ]:
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} column')
print(f'Date range: {df.index.min().date()}  to  {df.index.max().date()}')
print()

stats = df['discharge_cfs'].describe()
labels = {'count': 'Days of record', 'mean': 'Mean discharge',
          'std': 'Std deviation', 'min': 'Minimum', '25%': '25th percentile',
          '50%': 'Median', '75%': '75th percentile', 'max': 'Maximum'}
for key, label in labels.items():
    val = stats[key]
    unit = '' if key == 'count' else ' cfs'
    print(f'  {label:<20s}: {val:>12,.0f}{unit}')

## 4. Plotting Your First Hydrograph

A table of numbers obscures patterns that a plot reveals instantly.
Below we draw the full 15-year record, then zoom in on the 2011 flood event.

We divide discharge by 1,000 to plot in **kcfs** (thousands of cubic feet per second)
so the y-axis stays readable at flood magnitudes.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

ax.fill_between(df.index, df['discharge_cfs'] / 1_000, alpha=0.20, color='steelblue')
ax.plot(df.index, df['discharge_cfs'] / 1_000, color='steelblue', lw=0.8, alpha=0.9)

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Discharge (kcfs)', fontsize=11)
ax.set_title(
    'Missouri River at Omaha, NE Ã¢â‚¬â€ Daily Mean Discharge 2000Ã¢â‚¬â€œ2015\n'
    'USGS Gauge 06803495  |  Parameter 00060 (discharge, cfs)',
    fontsize=11,
)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
plt.tight_layout()
plt.show()

## 5. Zooming In: The 2011 Missouri River Flood

The spike visible around 2011 dwarfs everything else in the record.
The flood was triggered by an unusually wet spring combined with rapid Rocky Mountain
snowmelt that filled upstream reservoirs to capacity.
In June 2011 the Army Corps of Engineers opened emergency spillways, sending a sustained
pulse downstream that persisted for nearly **100 consecutive days above 500,000 cfs** Ã¢â‚¬â€
roughly 10Ãƒâ€” the median flow.

Let us isolate this window, annotate the peak, and compute some flood statistics.


In [ ]:
flood = df['2010-10-01':'2012-03-31']

peak_idx = flood['discharge_cfs'].idxmax()
peak_val = flood.loc[peak_idx, 'discharge_cfs']

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(flood.index, flood['discharge_cfs'] / 1_000, alpha=0.25, color='crimson')
ax.plot(flood.index, flood['discharge_cfs'] / 1_000, color='crimson', lw=1.5)
ax.axvline(peak_idx, color='darkred', ls='--', lw=1.8,
           label=f'Peak: {peak_val:,.0f} cfs  ({peak_idx.date()})')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Discharge (kcfs)', fontsize=11)
ax.set_title(
    'Missouri River Flood of 2011 Ã¢â‚¬â€ Gauge 06803495\n'
    'Oct 2010 Ã¢â‚¬â€œ Mar 2012',
    fontsize=11,
)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
plt.tight_layout()
plt.show()

days_above = (flood['discharge_cfs'] > 500_000).sum()
print(f'Peak discharge  : {peak_val:>12,.0f} cfs')
print(f'Date of peak    : {peak_idx.date()}')
print(f'Days above 500k : {days_above} days')
print(f'Median (full rec): {df["discharge_cfs"].median():>12,.0f} cfs')

## Key Takeaways

- A **hydrograph** records discharge over time, revealing how a watershed responds to
  precipitation and snowmelt through its rising limb, peak, and recession.
- **USGS NWIS** is the primary US streamflow database with 8,000+ active gauges;
  `dataretrieval.nwis.get_dv()` fetches daily values with a single function call.
- Real records contain extreme **interannual variability**: the 2011 Missouri River Flood
  exceeded 10Ãƒâ€” median flow Ã¢â‚¬â€ a pattern invisible in any summary statistic, but immediately
  apparent in the hydrograph.
- **Pandas DatetimeIndex** slicing (`df['2011':'2011']`) is the natural tool for isolating
  hydrological events; `idxmax()` locates the peak in one line.


## Exercise

**Try it yourself:**  
Filter the Missouri River dataset to **water year 2005**
(1 October 2004 Ã¢â‚¬â€œ 30 September 2005).
Plot the hydrograph for that water year and add a **horizontal dashed line** at the
peak discharge, labelled with the date and value in cfs.

*Hint: use `df['2004-10-01':'2005-09-30']` to slice by date string,
`ax.axhline(y, ls='--')` for the horizontal line,
and `.idxmax()` to find the date of peak discharge.*


In [ ]:
# Solution Ã¢â‚¬â€ try on your own first!
wy2005 = df['2004-10-01':'2005-09-30']
peak_idx = wy2005['discharge_cfs'].idxmax()
peak_val = wy2005.loc[peak_idx, 'discharge_cfs']

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(wy2005.index, wy2005['discharge_cfs'] / 1_000,
                alpha=0.25, color='steelblue')
ax.plot(wy2005.index, wy2005['discharge_cfs'] / 1_000,
        color='steelblue', lw=1.8, label='Daily streamflow')
ax.axhline(peak_val / 1_000, color='navy', ls='--', lw=1.5,
           label=f'Peak: {peak_val:,.0f} cfs  ({peak_idx.date()})')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Discharge (kcfs)', fontsize=11)
ax.set_title(
    'Missouri River at Omaha Ã¢â‚¬â€ Water Year 2005\n'
    '(1 Oct 2004 Ã¢â‚¬â€œ 30 Sep 2005)',
    fontsize=11,
)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
plt.tight_layout()
plt.show()